# Supervised Learning — Complete Masterclass Notes
## Prodigy Training Hub

**Trainer:** Kajola Gbenga  
**Title:** CEO, Prodigy Training Hub  
**Programme:** Data Science & Machine Learning Masterclass  

---

This notebook covers every major supervised learning algorithm with:
- When and why to use each algorithm
- Diagrammatic explanations
- Python code (scikit-learn, XGBoost, Keras)
- Interpretation of results
- Real-world Nigerian and global use cases

---

### Table of Contents
1. [What is Supervised Learning?](#1-what-is-supervised-learning)
2. [Linear Regression](#2-linear-regression)
3. [Logistic Regression](#3-logistic-regression)
4. [K-Nearest Neighbors (KNN)](#4-k-nearest-neighbors-knn)
5. [Decision Trees](#5-decision-trees)
6. [Random Forest](#6-random-forest)
7. [Support Vector Machines (SVM)](#7-support-vector-machines-svm)
8. [Naive Bayes](#8-naive-bayes)
9. [Gradient Boosting (XGBoost / LightGBM)](#9-gradient-boosting-xgboost--lightgbm)
10. [Neural Networks (MLP)](#10-neural-networks-mlp)
11. [Model Evaluation — Complete Reference](#11-model-evaluation--complete-reference)


## 0. Environment Setup

Run this cell first to install and import all required libraries.

In [143]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for notebooks without display
# Install required libraries (run once)
# !pip install scikit-learn xgboost lightgbm tensorflow matplotlib seaborn pandas numpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn core
from sklearn.model_selection import (train_test_split, cross_val_score,
    GridSearchCV, StratifiedKFold, KFold)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    mean_squared_error, mean_absolute_error,
    mean_absolute_percentage_error, r2_score
)

# Set consistent plot style
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
COLORS = {'blue': '#378ADD', 'coral': '#D85A30', 'teal': '#1D9E75',
          'amber': '#BA7517', 'purple': '#534AB7', 'green': '#639922'}

print("All libraries loaded successfully.")
print("Prodigy Training Hub | Trainer: Kajola Gbenga | CEO")


All libraries loaded successfully.
Prodigy Training Hub | Trainer: Kajola Gbenga | CEO


---
## 1. What is Supervised Learning?

Supervised learning trains a model on **labeled data** — each input example (X) comes with a known correct output (y).  
The model learns a mapping function **f(X) → y**, then predicts outputs for new, unseen data.

### Two main task types

| Task | Target type | Example | Metric |
|---|---|---|---|
| **Regression** | Continuous number | Salary, house price | RMSE, R² |
| **Classification** | Discrete class/label | Spam or Not, Disease Yes/No | Accuracy, F1, AUC |

### Supervised Learning Workflow

```
Labeled Data  →  Train/Test Split  →  Choose Algorithm  →  Fit Model
                                                                  ↓
                      Predict on New Data  ←  Evaluate (metrics)
```

### Key terminology
- **Features (X):** Input variables used to make predictions
- **Target (y):** The variable we are trying to predict
- **Training set:** Data used to fit the model (typically 80%)
- **Test set:** Held-out data used to evaluate generalization (typically 20%)
- **Overfitting:** Model memorizes training data, fails on new data
- **Underfitting:** Model is too simple to capture the pattern


In [144]:
# Visualise the Bias-Variance Tradeoff
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

np.random.seed(42)
x = np.linspace(0, 3, 100)
y_true = np.sin(x * 2) + 0.3 * x

x_pts = np.random.uniform(0, 3, 20)
y_pts  = np.sin(x_pts * 2) + 0.3 * x_pts + np.random.normal(0, 0.3, 20)

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

titles = ['Underfitting\n(too simple — degree 1)',
          'Good Fit\n(degree 4)',
          'Overfitting\n(too complex — degree 15)']
degrees = [1, 4, 15]
colors  = [COLORS['coral'], COLORS['teal'], COLORS['purple']]

for ax, deg, title, color in zip(axes, degrees, titles, colors):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(x_pts.reshape(-1,1), y_pts)
    ax.scatter(x_pts, y_pts, color='gray', alpha=0.6, s=30, label='Data')
    ax.plot(x, model.predict(x.reshape(-1,1)), color=color, lw=2, label=f'Degree {deg}')
    ax.plot(x, y_true, 'k--', lw=1, alpha=0.4, label='True pattern')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Feature X')
    ax.set_ylabel('Target y')
    ax.legend(fontsize=8)

plt.suptitle('Bias-Variance Tradeoff | Prodigy Training Hub', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('bias_variance.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()


---
## 2. Linear Regression

### What is it?
Linear Regression predicts a **continuous output** by fitting the best straight line (or hyperplane) through data points.  
It is the simplest and most interpretable regression algorithm.

### When to use
- Target variable is **continuous** (salary, revenue, price, score)
- Relationship between features and target is approximately **linear**
- You need **interpretable coefficients** (e.g. for stakeholder reports)
- Use as a **baseline** for all regression tasks

### Why it works
Minimizes the **Residual Sum of Squares (OLS)**:  
Each coefficient (β) tells you how much y changes per unit increase in X, holding all other features constant.

### Formula
```
ŷ = β₀ + β₁X₁ + β₂X₂ + ... + βₙXₙ

Loss (OLS) = Σ(yᵢ - ŷᵢ)²  →  Minimize
```

### How to interpret results
| Output | Interpretation |
|---|---|
| **R² (R-squared)** | Proportion of variance explained. R²=1 is perfect, R²=0 is baseline |
| **Coefficient βᵢ** | For every 1-unit increase in Xᵢ, y changes by βᵢ (all else equal) |
| **RMSE** | Average prediction error in the same units as y. Lower = better |
| **p-value** | p < 0.05 → feature is statistically significant |

### Regularisation Variants
| Variant | Penalty | When to use |
|---|---|---|
| OLS (plain) | None | Few features, no multicollinearity |
| Ridge | L2 = Σβ² | Multicollinearity present |
| Lasso | L1 = Σ|β| | Feature selection (drives irrelevant coefs to 0) |
| ElasticNet | L1 + L2 | Many features, mix of Ridge and Lasso |

### Real-World Use Cases (Nigerian context)
- Predicting employee salary from years of experience and education at GTBank / UBA
- Forecasting monthly revenue for NovaTrade retail branches
- Estimating Lagos property prices from size, location, and amenities
- Predicting MUSC Health ED patient volume from historical patterns


In [145]:
# ── LINEAR REGRESSION — Full Pipeline ──────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# Nigerian HR example: predict salary from experience and education
np.random.seed(42)
n = 200
years_exp   = np.random.randint(1, 20, n)
edu_level   = np.random.randint(1, 4, n)   # 1=BSc, 2=MSc, 3=PhD
salary      = (years_exp * 22000 + edu_level * 35000
               + np.random.normal(0, 15000, n))

df = pd.DataFrame({'years_experience': years_exp,
                   'education_level':  edu_level,
                   'salary_naira':     salary})

X = df[['years_experience', 'education_level']]
y = df['salary_naira']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Fit and evaluate all variants
models = {
    'OLS Linear': LinearRegression(),
    'Ridge (L2)': Ridge(alpha=1.0),
    'Lasso (L1)': Lasso(alpha=100),
    'ElasticNet': ElasticNet(alpha=100, l1_ratio=0.5)
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    results[name] = {
        'R²':   round(r2_score(y_test, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred)), 0),
        'MAE':  round(mean_absolute_error(y_test, y_pred), 0)
    }

results_df = pd.DataFrame(results).T
print("=" * 50)
print("Linear Regression Variants — Comparison")
print("Prodigy Training Hub | Trainer: Kajola Gbenga")
print("=" * 50)
print(results_df.to_string())

# OLS Coefficients
ols = models['OLS Linear']
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': ols.coef_})
print("\nOLS Coefficients (standardised):")
print(coef_df.to_string(index=False))
print("\nInterpretation: Higher coefficient = stronger influence on salary")


Linear Regression Variants — Comparison
Prodigy Training Hub | Trainer: Kajola Gbenga
                R²      RMSE       MAE
OLS Linear  0.9789   19719.0   14723.0
Ridge (L2)  0.9789   19741.0   14678.0
Lasso (L1)  0.9789   19723.0   14728.0
ElasticNet  0.0307  133800.0  112857.0

OLS Coefficients (standardised):
         Feature   Coefficient
years_experience 120263.902168
 education_level  27286.494597

Interpretation: Higher coefficient = stronger influence on salary


In [146]:
# ── Visualise: Actual vs Predicted + Residuals ──────────────────────────────
ols = models['OLS Linear']
y_pred_ols = ols.predict(X_test_sc)
residuals  = y_test.values - y_pred_ols

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Actual vs Predicted
axes[0].scatter(y_test, y_pred_ols, alpha=0.6, color=COLORS['blue'], s=30)
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual Salary (₦)'); axes[0].set_ylabel('Predicted Salary (₦)')
axes[0].set_title('Actual vs Predicted'); axes[0].legend()

# 2. Residuals vs Fitted
axes[1].scatter(y_pred_ols, residuals, alpha=0.6, color=COLORS['coral'], s=30)
axes[1].axhline(0, color='black', lw=1, linestyle='--')
axes[1].set_xlabel('Fitted Values'); axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot\n(should be random scatter around 0)')

# 3. Residual distribution
axes[2].hist(residuals, bins=20, color=COLORS['teal'], alpha=0.8, edgecolor='white')
axes[2].axvline(0, color='black', lw=1.5, linestyle='--')
axes[2].set_xlabel('Residual'); axes[2].set_ylabel('Frequency')
axes[2].set_title('Residual Distribution\n(should be approx. Normal)')

plt.suptitle('Linear Regression Diagnostics | Prodigy Training Hub', y=1.02)
plt.tight_layout()
plt.savefig('linear_regression_diagnostics.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print(f"R² = {r2_score(y_test, y_pred_ols):.4f} | RMSE = ₦{np.sqrt(mean_squared_error(y_test, y_pred_ols)):,.0f}")


R² = 0.9789 | RMSE = ₦19,719


---
## 3. Logistic Regression

### What is it?
Despite the name, Logistic Regression is a **classification** algorithm.  
It estimates the **probability** of an observation belonging to a class using the sigmoid function, then applies a threshold (default 0.5) to assign a label.

### When to use
- Binary or multi-class classification
- You need **probability estimates** (not just class labels)
- **Interpretable results** required (log-odds/odds ratios)
- When data is approximately **linearly separable**
- Best practice: always use as a **baseline classifier**

### Why it works
The sigmoid function squashes any linear combination of features into [0,1]:

```
P(y=1 | X) = σ(z) = 1 / (1 + e^(-z))
z = β₀ + β₁X₁ + ... + βₙXₙ

Loss = -[y·log(ŷ) + (1-y)·log(1-ŷ)]   ← cross-entropy
```

### How to interpret results
| Output | Interpretation |
|---|---|
| **Coefficient** | In log-odds. Exponentiate (e^β) to get odds ratio |
| **Odds ratio > 1** | Feature increases probability of class 1 |
| **Odds ratio < 1** | Feature decreases probability of class 1 |
| **AUC-ROC** | 0.5 = random, 1.0 = perfect. AUC > 0.8 is very good |
| **Precision** | Of all predicted positives, how many are truly positive? |
| **Recall** | Of all actual positives, how many did we catch? |

### Real-World Use Cases
- GTBank / First Bank Nigeria: Credit default prediction
- LUTH Hospital: Disease diagnosis (patient has condition or not)
- UBA Compliance: Suspicious transaction flag (fraud or not fraud)
- Email spam detection
- Customer churn prediction (will this customer leave in 30 days?)


In [147]:
# ── LOGISTIC REGRESSION — GTBank Credit Default ──────────────────────────────
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
n = 800
credit_score    = np.random.randint(300, 850, n)
income          = np.random.randint(50000, 800000, n)
loan_amount     = np.random.randint(100000, 5000000, n)
months_employed = np.random.randint(1, 120, n)
# Create realistic default labels
prob_default = 1 / (1 + np.exp(0.005*(credit_score - 600)
                                + 0.000001*income
                                - 0.01*months_employed))
defaulted = (np.random.rand(n) < prob_default).astype(int)

df_bank = pd.DataFrame({
    'credit_score':    credit_score,
    'income_naira':    income,
    'loan_amount':     loan_amount,
    'months_employed': months_employed,
    'defaulted':       defaulted
})

print(f"Dataset: {len(df_bank)} customers | Default rate: {defaulted.mean():.1%}")

X = df_bank.drop('defaulted', axis=1)
y = df_bank['defaulted']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_sc, y_train)

y_pred  = log_reg.predict(X_test_sc)
y_proba = log_reg.predict_proba(X_test_sc)[:, 1]

print("\n" + "="*55)
print("Classification Report — Credit Default Prediction")
print("="*55)
print(classification_report(y_test, y_pred,
      target_names=['No Default', 'Default']))

print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")

# Odds Ratios
odds_ratios = pd.Series(np.exp(log_reg.coef_[0]), index=X.columns)
print("\nOdds Ratios:")
print(odds_ratios.round(4).to_string())
print("\nOdds > 1 → increases probability of default")
print("Odds < 1 → decreases probability of default")


Dataset: 800 customers | Default rate: 55.6%

Classification Report — Credit Default Prediction
              precision    recall  f1-score   support

  No Default       0.66      0.55      0.60        71
     Default       0.68      0.78      0.73        89

    accuracy                           0.68       160
   macro avg       0.67      0.66      0.66       160
weighted avg       0.67      0.68      0.67       160

AUC-ROC: 0.7136

Odds Ratios:
credit_score       0.4318
income_naira       0.7833
loan_amount        0.9311
months_employed    1.2841

Odds > 1 → increases probability of default
Odds < 1 → decreases probability of default


In [148]:
# ── Confusion Matrix + ROC Curve ────────────────────────────────────────────
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Default', 'Default'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_score   = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color=COLORS['blue'], lw=2,
             label=f'AUC = {auc_score:.4f}')
axes[1].plot([0,1],[0,1],'k--', lw=1, label='Random (AUC=0.5)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color=COLORS['blue'])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

# 3. Predicted probability distribution
axes[2].hist(y_proba[y_test==0], bins=25, alpha=0.7,
             color=COLORS['teal'], label='No Default', density=True)
axes[2].hist(y_proba[y_test==1], bins=25, alpha=0.7,
             color=COLORS['coral'], label='Default', density=True)
axes[2].axvline(0.5, color='black', lw=1.5, linestyle='--', label='Threshold=0.5')
axes[2].set_xlabel('Predicted Probability of Default')
axes[2].set_ylabel('Density')
axes[2].set_title('Probability Distribution by Class')
axes[2].legend()

plt.suptitle('Logistic Regression Evaluation | Prodigy Training Hub', y=1.02)
plt.tight_layout()
plt.savefig('logistic_regression_eval.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()


---
## 4. K-Nearest Neighbors (KNN)

### What is it?
A non-parametric, instance-based algorithm. To predict a new point, KNN finds the K closest training points (by distance) and:
- **Classification:** takes a majority vote among K neighbors
- **Regression:** takes the mean (or weighted mean) of K neighbors' y-values

No training phase — all computation happens at prediction time ("lazy learner").

### When to use
- Small to medium datasets (slow on large data)
- Complex, non-linear relationships
- Recommendation systems and image similarity
- When you have no assumptions about data distribution

### Avoid when
- Very large dataset (prediction is O(n) — slow)
- High-dimensional features (distance becomes unreliable — "curse of dimensionality")
- Need model interpretability

### Why it works
Based on the assumption: **similar inputs produce similar outputs.**  
"Tell me who your neighbors are and I'll tell you who you are."

### The K tradeoff
| K | Bias | Variance | Risk |
|---|---|---|---|
| K=1 | Low | High | Overfitting — memorizes noise |
| K=3–10 | Balanced | Balanced | Usually best range |
| K very large | High | Low | Underfitting — too smooth |

### Key requirement: Feature scaling is MANDATORY
KNN uses Euclidean distance. Without scaling, features with large ranges dominate.

### Real-World Use Cases
- Product recommendation: "customers who bought X also bought Y"
- Image similarity search (find similar product images on Jumia)
- Anomaly detection in small sensor datasets
- Patient similarity in clinical records


In [149]:
# ── K-NEAREST NEIGHBORS — Elbow Method + Full Pipeline ─────────────────────
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1200, n_features=8,
                            n_informative=4, random_state=42)
feature_names = ['Age', 'Income', 'CreditScore', 'Tenure',
                 'TransactionFreq', 'AccountBalance',
                 'MonthsActive', 'NumProducts']
X = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── Elbow Method: find best K ────────────────────────────────────────────────
k_range      = range(1, 31)
train_errors = []
test_errors  = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_sc, y_train)
    train_errors.append(1 - knn.score(X_train_sc, y_train))
    test_errors.append(1 - knn.score(X_test_sc, y_test))

best_k = list(k_range)[np.argmin(test_errors)]
print(f"Best K = {best_k}  |  Test error = {min(test_errors):.4f}")

# Plot elbow
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(k_range, train_errors, 'o-', color=COLORS['coral'],
             lw=2, label='Training error')
axes[0].plot(k_range, test_errors, 's-', color=COLORS['blue'],
             lw=2, label='Test error')
axes[0].axvline(best_k, color=COLORS['teal'], lw=2, linestyle='--',
                label=f'Best K={best_k}')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Error rate')
axes[0].set_title('Elbow Method — Choosing Best K')
axes[0].legend()

# Final model
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_sc, y_train)
y_pred = knn_best.predict(X_test_sc)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title(f'KNN (K={best_k}) — Confusion Matrix')

plt.suptitle('KNN Analysis | Prodigy Training Hub', y=1.02)
plt.tight_layout()
plt.savefig('knn_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")


Best K = 3  |  Test error = 0.0833

Accuracy: 0.9167
F1 Score: 0.9174


---
## 5. Decision Trees

### What is it?
A tree-shaped model that splits data based on feature thresholds. Creates a flowchart of decision rules.  
Extremely **interpretable** — you can read the rules aloud in plain English.

### When to use
- **Interpretability is critical** (banking, healthcare, compliance, legal)
- Mix of categorical and numerical features
- No feature scaling required
- Want to visualize explicit decision rules
- Also used as **base learners** inside Random Forest and Gradient Boosting

### Why it works
At each node, the algorithm searches for the best feature + threshold that maximally **reduces impurity**:
- **Gini impurity** (default for classification): measures how often a randomly chosen element would be incorrectly classified
- **Entropy / Information Gain**: measures reduction in information disorder
- **MSE** (for regression trees): minimize variance within each leaf

```
Gini = 1 - Σ pᵢ²
Information Gain = Entropy(parent) - Σ weighted Entropy(children)
```

### How to interpret results
| Output | Interpretation |
|---|---|
| **Feature importance** | How much each feature reduces impurity on average |
| **max_depth** | Deeper tree = more complex rules = risk of overfitting |
| **export_text()** | Human-readable decision rules (perfect for compliance docs) |

### Preventing overfitting
- `max_depth`: limit how deep the tree grows (start with 3–5)
- `min_samples_split`: minimum samples required to split a node
- `min_samples_leaf`: minimum samples required at a leaf node

### Real-World Use Cases
- Banking: Explainable credit scoring rules required by CBN regulators
- Insurance: Policy pricing tiers based on risk factors
- LUTH Hospital: Clinical decision support (explainable differential diagnosis)
- Lagos State Government: Audit flag rules for procurement compliance


In [150]:
# ── DECISION TREES — Loan Approval + Feature Importance ─────────────────────
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

# GTBank loan approval dataset
np.random.seed(42)
n = 600
credit_score     = np.random.randint(300, 850, n)
income           = np.random.randint(50000, 800000, n)
employed_months  = np.random.randint(1, 120, n)
loan_amount_arr  = np.random.randint(100000, 5000000, n)
loan_to_income   = loan_amount_arr / income

# Decision rule embedded in labels
approved = ((credit_score >= 600) &
            (income >= 150000) &
            (employed_months >= 12)).astype(int)

df_tree = pd.DataFrame({
    'credit_score':    credit_score,
    'income_naira':    income,
    'employed_months': employed_months,
    'approved':        approved
})

X = df_tree.drop('approved', axis=1)
y = df_tree['approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Compare different depths
for depth in [2, 3, 5, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_acc = tree.score(X_train, y_train)
    test_acc  = tree.score(X_test, y_test)
    print(f"max_depth={str(depth):4s} | Train acc={train_acc:.4f} | "
          f"Test acc={test_acc:.4f} | Nodes={tree.tree_.node_count}")


max_depth=2    | Train acc=0.9688 | Test acc=0.9500 | Nodes=7
max_depth=3    | Train acc=1.0000 | Test acc=1.0000 | Nodes=11
max_depth=5    | Train acc=1.0000 | Test acc=1.0000 | Nodes=11
max_depth=None | Train acc=1.0000 | Test acc=1.0000 | Nodes=11


In [151]:
# ── Visualise Tree + Feature Importance ─────────────────────────────────────
tree_best = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_best.fit(X_train, y_train)
y_pred = tree_best.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Tree diagram
plot_tree(tree_best,
          feature_names=list(X.columns),
          class_names=['Decline', 'Approve'],
          filled=True, rounded=True,
          ax=axes[0], fontsize=9,
          impurity=True, proportion=False)
axes[0].set_title('Decision Tree (max_depth=3) — Loan Approval')

# 2. Feature importance
importance = pd.Series(tree_best.feature_importances_,
                       index=X.columns).sort_values()
axes[1].barh(importance.index, importance.values,
             color=[COLORS['blue'], COLORS['teal'], COLORS['coral']])
axes[1].set_xlabel('Gini Importance')
axes[1].set_title('Feature Importance')
for i, v in enumerate(importance.values):
    axes[1].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=10)

plt.suptitle('Decision Tree Analysis | Prodigy Training Hub', y=1.02)
plt.tight_layout()
plt.savefig('decision_tree_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# Human-readable rules for compliance
print("\n" + "="*55)
print("Decision Rules (export for compliance documentation)")
print("="*55)
print(export_text(tree_best, feature_names=list(X.columns)))
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")



Decision Rules (export for compliance documentation)
|--- credit_score <= 603.50
|   |--- credit_score <= 598.50
|   |   |--- class: 0
|   |--- credit_score >  598.50
|   |   |--- income_naira <= 289866.00
|   |   |   |--- class: 0
|   |   |--- income_naira >  289866.00
|   |   |   |--- class: 1
|--- credit_score >  603.50
|   |--- income_naira <= 152703.50
|   |   |--- class: 0
|   |--- income_naira >  152703.50
|   |   |--- employed_months <= 11.50
|   |   |   |--- class: 0
|   |   |--- employed_months >  11.50
|   |   |   |--- class: 1

Test Accuracy: 1.0000


---
## 6. Random Forest

### What is it?
An ensemble of many decision trees. Each tree is trained on a **random bootstrap sample** of rows, and each split considers only a **random subset of features**.  
Final prediction = majority vote (classification) or mean (regression).

### When to use
- Tabular data of any size
- When you need high accuracy and can sacrifice some interpretability
- When single decision trees are overfitting
- Handles missing values, outliers, and irrelevant features well
- One of the most reliable all-purpose algorithms available

### Why it works — two sources of randomness reduce variance
1. **Bagging** (Bootstrap Aggregating): each tree sees a different random sample of rows → different errors
2. **Feature randomness**: each split only considers √(n_features) random features → decorrelated trees
3. Averaging uncorrelated trees cancels out individual mistakes → lower variance

### How to interpret results
| Output | Interpretation |
|---|---|
| **Feature importance** | Gini-based: how much each feature reduces impurity across all trees |
| **OOB score** | Free validation without a test set (uses data each tree didn't see) |
| **n_estimators** | More trees = better (diminishing returns after ~300) |
| **max_features** | 'sqrt' for classification, 'sqrt' or 'log2' for regression |

### Real-World Use Cases
- Jumia Nigeria: Product category prediction from listing features
- LUTH Hospital: Patient readmission risk within 30 days
- Lagos State Government: Employee attrition early warning
- MUSC Health: ED staffing need prediction from historical patterns
- First Bank Nigeria: Anti-money laundering transaction flagging


In [152]:
# ── RANDOM FOREST — Full Pipeline + OOB + Feature Importance ────────────────
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)
from sklearn.datasets import make_classification
X_rf, y_rf = make_classification(n_samples=3000, n_features=12,
                                  n_informative=7, random_state=42)
feat_names = ['Age', 'Income', 'CreditScore', 'Tenure', 'TransactionFreq',
              'AccountBalance', 'NumProducts', 'HasCreditCard',
              'ActiveMember', 'Salary', 'CommuteDist', 'ContractType']
X_rf = pd.DataFrame(X_rf, columns=feat_names)

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf, y_rf, test_size=0.2, random_state=42)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    max_features='sqrt',
    oob_score=True,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_rf, y_train_rf)
y_pred_rf = rf.predict(X_test_rf)

print("="*55)
print("Random Forest — Employee Attrition Prediction")
print("Prodigy Training Hub | Trainer: Kajola Gbenga")
print("="*55)
print(f"OOB Score (free validation): {rf.oob_score_:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test_rf, y_pred_rf):.4f}")
print(f"F1 Score: {f1_score(y_test_rf, y_pred_rf):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test_rf, rf.predict_proba(X_test_rf)[:,1]):.4f}")
print("\n" + classification_report(y_test_rf, y_pred_rf))


Random Forest — Employee Attrition Prediction
Prodigy Training Hub | Trainer: Kajola Gbenga
OOB Score (free validation): 0.9300
Test Accuracy: 0.9467
F1 Score: 0.9458
AUC-ROC: 0.9859

              precision    recall  f1-score   support

           0       0.94      0.95      0.95       303
           1       0.95      0.94      0.95       297

    accuracy                           0.95       600
   macro avg       0.95      0.95      0.95       600
weighted avg       0.95      0.95      0.95       600



In [153]:
# ── n_estimators tradeoff + Feature Importance Visualization ────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. OOB error vs n_estimators
oob_errors = []
n_est_range = [10, 25, 50, 100, 150, 200, 300]
for n in n_est_range:
    rf_tmp = RandomForestClassifier(n_estimators=n, oob_score=True,
                                     n_jobs=-1, random_state=42)
    rf_tmp.fit(X_train_rf, y_train_rf)
    oob_errors.append(1 - rf_tmp.oob_score_)

axes[0].plot(n_est_range, oob_errors, 'o-', color=COLORS['blue'], lw=2)
axes[0].set_xlabel('Number of Trees'); axes[0].set_ylabel('OOB Error')
axes[0].set_title('OOB Error vs n_estimators\n(diminishing returns after ~150)')

# 2. Feature importance (Gini)
importance = pd.Series(rf.feature_importances_, index=feat_names).sort_values()
colors_bar  = [COLORS['coral'] if v > importance.median() else COLORS['blue']
               for v in importance.values]
axes[1].barh(importance.index, importance.values, color=colors_bar)
axes[1].set_xlabel('Gini Importance')
axes[1].set_title('Feature Importance (Gini)')
axes[1].axvline(importance.median(), color='gray', lw=1, linestyle='--',
                label='Median')
axes[1].legend()

# 3. Confusion matrix
cm_rf = confusion_matrix(y_test_rf, y_pred_rf)
ConfusionMatrixDisplay(cm_rf).plot(ax=axes[2], colorbar=False, cmap='Oranges')
axes[2].set_title('Confusion Matrix')

plt.suptitle('Random Forest Analysis | Prodigy Training Hub', y=1.02)
plt.tight_layout()
plt.savefig('random_forest_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()


---
## 7. Support Vector Machines (SVM)

### What is it?
Finds the **hyperplane** that maximally separates classes with the **widest possible margin**.  
Data points nearest to the boundary are called **support vectors**.  
Uses the **kernel trick** to handle non-linear separability.

### When to use
- High-dimensional data (text classification, bioinformatics)
- Small to medium datasets (< 100k rows — slow on large data)
- Need a powerful non-linear classifier (use RBF kernel)
- Clear margin of separation exists in the data

### Avoid when
- Dataset is very large (O(n²) or O(n³) training time)
- Need fast training and prediction (use LightGBM instead)
- Need native probability outputs (requires Platt scaling — extra cost)

### Why it works
By maximizing the margin, SVM finds the most **robust** decision boundary.  
The **kernel trick** maps data to a higher-dimensional space where it becomes linearly separable — without explicitly computing the transformation.

```
Maximize:  2/‖w‖   (margin width)
Subject to: yᵢ(w·xᵢ + b) ≥ 1  for all i

Kernel: K(xᵢ, xⱼ) = φ(xᵢ)·φ(xⱼ)
RBF Kernel: K(xᵢ, xⱼ) = exp(-γ‖xᵢ - xⱼ‖²)
```

### Key hyperparameters
| Parameter | Effect | Typical range |
|---|---|---|
| **C** | Regularization. Low C = wide margin, allow misclassifications. High C = narrow margin | 0.01 – 1000 |
| **gamma** (RBF) | Influence radius. Low = far reach (smooth). High = close (complex) | 'scale', 0.001–1 |
| **kernel** | Type of decision boundary | 'linear', 'rbf', 'poly' |

### Kernel guide
| Kernel | Use when |
|---|---|
| `linear` | Text data, many features, linearly separable |
| `rbf` | General purpose, most common, non-linear |
| `poly` | Image processing |
| `sigmoid` | Resembles neural network activation |

### Feature scaling is MANDATORY for SVM

### Real-World Use Cases
- Text classification: UBA transaction description categorization
- Image recognition: product image classification on Jumia
- Bioinformatics: gene expression classification
- Fraud detection in high-dimensional feature spaces


In [154]:
# ── SVM — Classification with Multiple Kernels ───────────────────────────────
from sklearn.svm import SVC

np.random.seed(42)
from sklearn.datasets import make_classification
X_svm, y_svm = make_classification(n_samples=600, n_features=6,
                                    n_informative=3, random_state=42)
X_train_sv, X_test_sv, y_train_sv, y_test_sv = train_test_split(
    X_svm, y_svm, test_size=0.2, random_state=42)

scaler_sv = StandardScaler()
X_tr_sv   = scaler_sv.fit_transform(X_train_sv)
X_te_sv   = scaler_sv.transform(X_test_sv)

kernels = ['linear', 'rbf', 'poly', 'sigmoid']
svm_results = {}

for kernel in kernels:
    svm = SVC(kernel=kernel, C=1.0, gamma='scale',
              probability=True, random_state=42)
    svm.fit(X_tr_sv, y_train_sv)
    y_pred_sv = svm.predict(X_te_sv)
    y_prob_sv = svm.predict_proba(X_te_sv)[:, 1]
    svm_results[kernel] = {
        'Accuracy': accuracy_score(y_test_sv, y_pred_sv),
        'F1':       f1_score(y_test_sv, y_pred_sv),
        'AUC':      roc_auc_score(y_test_sv, y_prob_sv)
    }

results_svm_df = pd.DataFrame(svm_results).T.round(4)
print("="*50)
print("SVM Kernel Comparison | Prodigy Training Hub")
print("="*50)
print(results_svm_df.to_string())
print("\nRecommendation: RBF is the default starting point for SVM.")


SVM Kernel Comparison | Prodigy Training Hub
         Accuracy      F1     AUC
linear     0.8667  0.8730  0.9381
rbf        0.9417  0.9421  0.9842
poly       0.8083  0.8296  0.9147
sigmoid    0.7750  0.7907  0.8319

Recommendation: RBF is the default starting point for SVM.


In [155]:
# ── SVM C parameter effect + 2D Decision Boundary ───────────────────────────
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)
X_sc_moons = StandardScaler().fit_transform(X_moons)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
C_values = [0.1, 1.0, 100.0]

for ax, C in zip(axes, C_values):
    svm = SVC(kernel='rbf', C=C, gamma='scale')
    svm.fit(X_sc_moons, y_moons)

    xx, yy = np.meshgrid(np.linspace(-3,3,200), np.linspace(-3,3,200))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')
    ax.scatter(X_sc_moons[:,0], X_sc_moons[:,1], c=y_moons,
               cmap='coolwarm', s=20, edgecolors='k', linewidths=0.3)
    acc = svm.score(X_sc_moons, y_moons)
    ax.set_title(f'SVM RBF | C={C} | Acc={acc:.3f}')
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

plt.suptitle('SVM: Effect of C Parameter | Prodigy Training Hub', y=1.02)
plt.tight_layout()
plt.savefig('svm_decision_boundary.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print("Low C → wide margin, some misclassifications allowed")
print("High C → narrow margin, tries to classify everything correctly (risk overfitting)")


Low C → wide margin, some misclassifications allowed
High C → narrow margin, tries to classify everything correctly (risk overfitting)


---
## 8. Naive Bayes

### What is it?
A probabilistic classifier based on **Bayes' Theorem** with the "naive" assumption that all features are **conditionally independent**.  
Despite the oversimplification, it performs remarkably well on text classification and is extremely fast.

### When to use
- **Text classification** (spam, sentiment, document categorization)
- **Real-time prediction** required (fastest classifier available)
- Very large datasets
- Small training data with many features (NLP)
- Feature independence holds approximately

### Why it works
Bayes' theorem:
```
P(class | features) ∝ P(class) × ∏ P(featureᵢ | class)
```
The naive independence assumption makes computation tractable — multiply individual probabilities instead of computing massive joint distributions.

### Variants
| Variant | Distribution assumed | Best for |
|---|---|---|
| **GaussianNB** | Normal (Gaussian) | Continuous features |
| **MultinomialNB** | Multinomial | Word counts, TF-IDF |
| **BernoulliNB** | Bernoulli | Binary features (word present/absent) |
| **ComplementNB** | Complement | Imbalanced text datasets |

### Laplace Smoothing (alpha)
Prevents zero probabilities for words unseen during training.  
Default `alpha=1.0` adds a small count to every word in every class.

### How to interpret results
- **Class prior probabilities**: P(class) — proportion of each class
- **Feature log probabilities**: log P(featureᵢ | class) — inspect to see which words drive each class
- Probabilities are often poorly calibrated — use for ranking, not exact estimates

### Real-World Use Cases
- Email spam filtering
- GTBank / UBA: Transaction category classification from description text
- LUTH Hospital: ICD code category prediction from clinical notes
- Customer complaint routing: which department handles this ticket?
- News article classification: politics vs business vs sports


In [156]:
# ── NAIVE BAYES — Text Spam Classifier + Gaussian NB ────────────────────────
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Example 1: Spam detection (MultinomialNB) ────────────────────────────────
messages = [
    "Win FREE money now click here",
    "Congratulations you have been selected for a prize",
    "Meeting agenda for tomorrow at 9am",
    "Dear colleague please review the attached report",
    "URGENT: Claim your reward today, limited time",
    "Free loan offer no credit check required",
    "Team standup call at 9am, please be on time",
    "Please review the Q3 quarterly financial report",
    "You have won a lottery prize, click to claim",
    "Project update: all milestones on track this week",
    "Double your income with this secret investment",
    "Invoice attached for your review and approval",
    "You are pre-approved for a credit card limit",
    "Reminder: performance review scheduled for Friday",
    "Act now: exclusive offer expires in 24 hours",
    "Hi team, please find attached the budget forecast",
    "Congratulations winner: claim your cash prize now",
    "Board meeting minutes from yesterday are attached",
]
labels = [1,1,0,0,1,1,0,0,1,0,1,0,1,0,1,0,1,0]  # 1=spam, 0=ham

X_train_nb, X_test_nb, y_train_nb, y_test_nb = train_test_split(
    messages, labels, test_size=0.25, random_state=42)

nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', lowercase=True,
                               ngram_range=(1,2))),
    ('nb',    MultinomialNB(alpha=1.0))
])
nb_pipeline.fit(X_train_nb, y_train_nb)
y_pred_nb = nb_pipeline.predict(X_test_nb)

print("="*55)
print("Naive Bayes — Email Spam Detection")
print("Prodigy Training Hub | Trainer: Kajola Gbenga")
print("="*55)
print(classification_report(y_test_nb, y_pred_nb,
      target_names=['Ham (legitimate)', 'Spam']))

# Test on new messages
test_msgs = [
    "Claim your free prize now, click here immediately",
    "Please review attached budget forecast for Q4",
    "Congratulations! Your account has been selected",
    "Team meeting rescheduled to 3pm on Thursday"
]
predictions = nb_pipeline.predict(test_msgs)
probabilities = nb_pipeline.predict_proba(test_msgs)
print("\nNew message predictions:")
print("-" * 55)
for msg, pred, prob in zip(test_msgs, predictions, probabilities):
    label = "SPAM" if pred == 1 else "HAM"
    conf  = max(prob)
    print(f"[{label:4s}] ({conf:.0%} confidence): {msg[:50]}...")


Naive Bayes — Email Spam Detection
Prodigy Training Hub | Trainer: Kajola Gbenga
                  precision    recall  f1-score   support

Ham (legitimate)       0.50      1.00      0.67         1
            Spam       1.00      0.75      0.86         4

        accuracy                           0.80         5
       macro avg       0.75      0.88      0.76         5
    weighted avg       0.90      0.80      0.82         5


New message predictions:
-------------------------------------------------------
[SPAM] (53% confidence): Claim your free prize now, click here immediately...
[HAM ] (78% confidence): Please review attached budget forecast for Q4...
[HAM ] (53% confidence): Congratulations! Your account has been selected...
[HAM ] (75% confidence): Team meeting rescheduled to 3pm on Thursday...


In [157]:
# ── GaussianNB for continuous features ──────────────────────────────────────
from sklearn.datasets import load_iris

iris     = load_iris()
X_iris   = iris.data
y_iris   = iris.target

X_tr_iris, X_te_iris, y_tr_iris, y_te_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42)

gnb = GaussianNB()
gnb.fit(X_tr_iris, y_tr_iris)
y_pred_gnb = gnb.predict(X_te_iris)

print("GaussianNB — Iris dataset (continuous features)")
print(f"Accuracy: {accuracy_score(y_te_iris, y_pred_gnb):.4f}")

# Class priors and feature means
print("\nClass prior probabilities:")
for cls, prior in zip(iris.target_names, gnb.class_prior_):
    print(f"  {cls}: {prior:.4f}")

print("\nFeature means per class (learned from data):")
means_df = pd.DataFrame(gnb.theta_,
                        index=iris.target_names,
                        columns=iris.feature_names)
print(means_df.round(3).to_string())
print("\nInterpretation: Higher mean for a class on a feature")
print("means that feature is diagnostic for distinguishing that class.")


GaussianNB — Iris dataset (continuous features)
Accuracy: 1.0000

Class prior probabilities:
  setosa: 0.3333
  versicolor: 0.3417
  virginica: 0.3250

Feature means per class (learned from data):
            sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
setosa                  4.990             3.452              1.450             0.245
versicolor              5.920             2.771              4.241             1.322
virginica               6.533             2.967              5.521             2.000

Interpretation: Higher mean for a class on a feature
means that feature is diagnostic for distinguishing that class.


---
## 9. Gradient Boosting (XGBoost / LightGBM)

### What is it?
Builds an ensemble of trees **sequentially**, where each new tree corrects the residual errors of the previous ensemble.  
The **gradient** refers to gradient descent — each tree is fit to the negative gradient of the loss function.  
XGBoost, LightGBM, and CatBoost are the most widely used implementations.

### When to use
- **Tabular structured data** (most common case)
- You need maximum predictive accuracy on a deadline
- Finance (fraud, credit scoring), healthcare, Kaggle competitions
- You have time and resources to tune hyperparameters
- Dataset is medium to large (thousands to millions of rows)

### Why it works — the boosting idea
```
Random Forest:  Build many independent trees, average predictions
Gradient Boost: Build trees SEQUENTIALLY, each correcting the previous ensemble

F₀(x) = base prediction (mean of y)
F₁(x) = F₀(x) + α × tree₁(x)   ← tree fits residuals of F₀
F₂(x) = F₁(x) + α × tree₂(x)   ← tree fits residuals of F₁
...
Fₙ(x) = F₀(x) + α Σ treeₜ(x)
```

### Key hyperparameters
| Parameter | Effect | Typical range |
|---|---|---|
| **n_estimators** | Number of trees. Use early stopping | 100 – 2000 |
| **learning_rate** | Step size. Lower = more robust, needs more trees | 0.01 – 0.3 |
| **max_depth** | Tree depth. Shallower = less overfit | 3 – 8 |
| **subsample** | Fraction of rows per tree | 0.6 – 0.9 |
| **colsample_bytree** | Fraction of features per tree | 0.6 – 0.9 |
| **scale_pos_weight** | Class imbalance correction ratio (XGBoost) | neg/pos count |

### XGBoost Feature Importance Types
| Type | Meaning |
|---|---|
| **weight** | Number of times feature appears in splits |
| **gain** | Average information gain per split (most reliable) |
| **cover** | Average number of samples affected per split |

### Real-World Use Cases
- GTBank / First Bank: Fraud detection (imbalanced dataset, high AUC required)
- Jumia: Customer purchase propensity scoring
- UBA Compliance: AML transaction risk scoring
- MUSC Health: Patient readmission and ED overcrowding prediction
- Insurance: Claim amount prediction (regression)


In [158]:
# ── XGBOOST — Fraud Detection with Class Imbalance ───────────────────────────
import xgboost as xgb

np.random.seed(42)
from sklearn.datasets import make_classification
X_xgb, y_xgb = make_classification(
    n_samples=8000, n_features=15, n_informative=10,
    weights=[0.97, 0.03],   # 3% fraud — heavily imbalanced
    random_state=42
)
feat_names_xgb = [f'feature_{i}' for i in range(X_xgb.shape[1])]

X_tr_xgb, X_te_xgb, y_tr_xgb, y_te_xgb = train_test_split(
    X_xgb, y_xgb, test_size=0.2, random_state=42, stratify=y_xgb)

print(f"Training set: {len(y_tr_xgb)} samples | Fraud rate: {y_tr_xgb.mean():.2%}")
scale_pos = (y_tr_xgb == 0).sum() / (y_tr_xgb == 1).sum()
print(f"scale_pos_weight = {scale_pos:.1f}  (handles class imbalance)")

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,   # key for imbalanced data
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_model.fit(X_tr_xgb, y_tr_xgb,
              eval_set=[(X_te_xgb, y_te_xgb)],
              verbose=False)

y_pred_xgb  = xgb_model.predict(X_te_xgb)
y_proba_xgb = xgb_model.predict_proba(X_te_xgb)[:, 1]

print("\n" + "="*55)
print("XGBoost — ATM Fraud Detection")
print("Prodigy Training Hub | Trainer: Kajola Gbenga")
print("="*55)
print(classification_report(y_te_xgb, y_pred_xgb,
      target_names=['Legitimate', 'Fraud']))
print(f"AUC-ROC: {roc_auc_score(y_te_xgb, y_proba_xgb):.4f}")
print(f"Best iteration: {xgb_model.best_iteration}")


Training set: 6400 samples | Fraud rate: 3.34%
scale_pos_weight = 28.9  (handles class imbalance)

XGBoost — ATM Fraud Detection
Prodigy Training Hub | Trainer: Kajola Gbenga
              precision    recall  f1-score   support

  Legitimate       0.99      0.99      0.99      1546
       Fraud       0.61      0.63      0.62        54

    accuracy                           0.97      1600
   macro avg       0.80      0.81      0.80      1600
weighted avg       0.97      0.97      0.97      1600

AUC-ROC: 0.9321
Best iteration: 10


In [159]:
# ── XGBoost Feature Importance + Learning Curve ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Feature importance (gain)
importance_gain = pd.Series(
    xgb_model.get_booster().get_score(importance_type='gain')
).sort_values(ascending=True).tail(10)
axes[0].barh(importance_gain.index, importance_gain.values,
             color=COLORS['blue'])
axes[0].set_xlabel('Gain'); axes[0].set_title('Top 10 Features (Gain)')

# 2. Learning curves (training AUC vs eval AUC)
results = xgb_model.evals_result()
epochs  = len(results['validation_0']['auc'])
axes[1].plot(range(epochs), results['validation_0']['auc'],
             color=COLORS['teal'], lw=2, label='Eval AUC')
axes[1].axvline(xgb_model.best_iteration, color='red',
                lw=1.5, linestyle='--',
                label=f'Best iter={xgb_model.best_iteration}')
axes[1].set_xlabel('Boosting Round')
axes[1].set_ylabel('AUC')
axes[1].set_title('Learning Curve')
axes[1].legend()

# 3. Precision-Recall curve (better than ROC for imbalanced)
from sklearn.metrics import precision_recall_curve, average_precision_score
prec, rec, _ = precision_recall_curve(y_te_xgb, y_proba_xgb)
ap = average_precision_score(y_te_xgb, y_proba_xgb)
axes[2].plot(rec, prec, color=COLORS['coral'], lw=2, label=f'AP={ap:.4f}')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve\n(use for imbalanced datasets)')
axes[2].legend()

plt.suptitle('XGBoost Analysis | Prodigy Training Hub', y=1.02)
plt.tight_layout()
plt.savefig('xgboost_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()


---
## 10. Neural Networks (MLP)

### What is it?
A Multi-Layer Perceptron (MLP) consists of **layers of interconnected neurons**.  
Input features flow forward through hidden layers, each applying a weighted sum and activation function.  
Parameters are updated via **backpropagation** and **gradient descent**.

### When to use
- Complex **non-linear relationships** that tree models can't capture
- **Image data, text data, audio** (with deep architectures)
- Very large datasets (millions of rows)
- When feature interactions are too complex for manual engineering

### Avoid when
- Small dataset (< 1000 samples) — trees generalize better
- Interpretability is required (black box)
- No GPU resources and dataset is large
- Training time is critical

### Architecture
```
Input Layer  →  Hidden Layer 1  →  Hidden Layer 2  →  Output Layer
  (features)       (ReLU)              (ReLU)        (Sigmoid/Softmax)
```

### Activation Functions
| Function | Formula | Use in |
|---|---|---|
| **ReLU** | max(0, z) | Hidden layers (default) |
| **Sigmoid** | 1/(1+e^(-z)) | Binary output |
| **Softmax** | e^z / Σe^z | Multi-class output |
| **Tanh** | (e^z - e^(-z))/(e^z + e^(-z)) | Hidden (when negative features matter) |

### Training tips
- Always **scale features** (StandardScaler or MinMaxScaler)
- Use **Dropout** (0.2–0.5) to prevent overfitting
- Use **BatchNormalization** for faster, more stable training
- Use **EarlyStopping** (patience=10–20) to prevent overtraining
- Use **ReduceLROnPlateau** to automatically reduce learning rate on plateau

### How to interpret training
| Signal | Meaning |
|---|---|
| Training loss falls, validation loss rises | Overfitting — add Dropout or get more data |
| Both losses plateau early | Underfitting — increase model capacity or train longer |
| Validation loss oscillates | Learning rate too high — reduce it |
| Both losses decrease smoothly | Good training — ideal pattern |

### Real-World Use Cases
- Image classification (product quality control)
- Customer behaviour modelling from high-dimensional features
- NLP text classification with embedding layers
- Time-series forecasting with LSTM (advanced)


In [160]:
# ── NEURAL NETWORK — sklearn MLPClassifier ───────────────────────────────────
from sklearn.neural_network import MLPClassifier

np.random.seed(42)
from sklearn.datasets import make_classification
X_nn, y_nn = make_classification(n_samples=3000, n_features=14,
                                   n_informative=9, random_state=42)

X_tr_nn, X_te_nn, y_tr_nn, y_te_nn = train_test_split(
    X_nn, y_nn, test_size=0.2, random_state=42)

scaler_nn  = StandardScaler()
X_tr_nn_sc = scaler_nn.fit_transform(X_tr_nn)
X_te_nn_sc = scaler_nn.transform(X_te_nn)

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),  # 3 hidden layers
    activation='relu',
    solver='adam',
    alpha=0.001,                        # L2 regularisation
    learning_rate_init=0.001,
    batch_size=64,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42,
    verbose=False
)
mlp.fit(X_tr_nn_sc, y_tr_nn)

y_pred_nn  = mlp.predict(X_te_nn_sc)
y_proba_nn = mlp.predict_proba(X_te_nn_sc)[:, 1]

print("="*55)
print("Neural Network (MLP) — Classification")
print("Prodigy Training Hub | Trainer: Kajola Gbenga")
print("="*55)
print(classification_report(y_te_nn, y_pred_nn))
print(f"AUC-ROC: {roc_auc_score(y_te_nn, y_proba_nn):.4f}")
print(f"Best validation score: {mlp.best_validation_score_:.4f}")
print(f"Stopped at iteration: {mlp.n_iter_}")


Neural Network (MLP) — Classification
Prodigy Training Hub | Trainer: Kajola Gbenga
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       297
           1       0.97      0.95      0.96       303

    accuracy                           0.96       600
   macro avg       0.96      0.96      0.96       600
weighted avg       0.96      0.96      0.96       600

AUC-ROC: 0.9885
Best validation score: 0.9583
Stopped at iteration: 40


In [161]:
# ── Training Loss Curve + Architecture Comparison ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# 1. Loss curve
axes[0].plot(mlp.loss_curve_, color=COLORS['blue'], lw=2, label='Training loss')
if hasattr(mlp, 'validation_scores_'):
    val_loss = [-s for s in mlp.validation_scores_]
    axes[0].plot(val_loss, color=COLORS['coral'], lw=2, linestyle='--',
                 label='Validation loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Curve')
axes[0].legend()

# 2. Compare architectures
arch_results = {}
architectures = {
    'Shallow (64)':         (64,),
    'Medium (128,64)':      (128, 64),
    'Deep (128,64,32)':     (128, 64, 32),
    'Very Deep (256,128,64,32)': (256, 128, 64, 32)
}
for name, arch in architectures.items():
    m = MLPClassifier(hidden_layer_sizes=arch, max_iter=200,
                      random_state=42, early_stopping=True,
                      validation_fraction=0.1, verbose=False)
    m.fit(X_tr_nn_sc, y_tr_nn)
    acc = accuracy_score(y_te_nn, m.predict(X_te_nn_sc))
    arch_results[name] = acc

arch_df = pd.Series(arch_results).sort_values()
bars = axes[1].barh(arch_df.index, arch_df.values,
                    color=[COLORS['blue'], COLORS['teal'],
                           COLORS['coral'], COLORS['purple']])
axes[1].set_xlabel('Test Accuracy')
axes[1].set_title('Architecture Comparison')
for bar, val in zip(bars, arch_df.values):
    axes[1].text(val + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)

# 3. Confusion matrix
cm_nn = confusion_matrix(y_te_nn, y_pred_nn)
ConfusionMatrixDisplay(cm_nn).plot(ax=axes[2], colorbar=False, cmap='Purples')
axes[2].set_title('Confusion Matrix')

plt.suptitle('Neural Network Analysis | Prodigy Training Hub', y=1.02)
plt.tight_layout()
plt.savefig('neural_network_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()


---
## 11. Model Evaluation — Complete Reference

### The Confusion Matrix (Binary Classification)

```
                 Predicted Positive    Predicted Negative
Actual Positive   True Positive (TP)   False Negative (FN)   ← Type II error
Actual Negative   False Positive (FP)  True Negative (TN)    ← Type I error
```

### Classification Metrics
| Metric | Formula | Use when |
|---|---|---|
| **Accuracy** | (TP+TN)/N | Balanced classes |
| **Precision** | TP/(TP+FP) | Cost of false positives is high (spam, content moderation) |
| **Recall** | TP/(TP+FN) | Cost of false negatives is high (disease detection, fraud) |
| **F1 Score** | 2PR/(P+R) | Imbalanced data, need balance of Precision and Recall |
| **AUC-ROC** | Area under ROC curve | Probability ranking quality |
| **AP Score** | Area under Precision-Recall | Severely imbalanced data (fraud detection) |
| **Cohen's Kappa** | (Acc−Base)/(1−Base) | Multi-class, accounts for chance agreement |

### Regression Metrics
| Metric | Formula | Notes |
|---|---|---|
| **R² (R-squared)** | 1 − SS_res/SS_tot | 1=perfect, 0=mean baseline. Can be negative |
| **RMSE** | √(Σ(y−ŷ)²/n) | Same units as target. Penalizes large errors |
| **MAE** | Σ|y−ŷ|/n | More robust to outliers than RMSE |
| **MAPE** | Σ|y−ŷ|/y × 100% | Scale-independent %, undefined when y=0 |

### Cross-Validation Types
| Method | When to use |
|---|---|
| **K-Fold (k=5 or 10)** | General purpose — most common |
| **Stratified K-Fold** | Classification with imbalanced classes |
| **TimeSeriesSplit** | Time-series — no data leakage from future |
| **LOOCV** | Very small datasets (expensive) |

### Final Algorithm Selection Guide
| Algorithm | Interpretable | Scales | Scaling needed | Best for |
|---|---|---|---|---|
| Linear Regression | High | Yes | Recommended | Baseline regression, explainability |
| Logistic Regression | High | Yes | Yes | Baseline classifier, probability estimates |
| KNN | Low | No | Mandatory | Recommendation, small datasets |
| Decision Tree | Very High | Moderate | No | Compliance, rule extraction |
| Random Forest | Moderate | Yes | No | General-purpose tabular data |
| SVM | Low | No | Mandatory | Text, high-dimensional, small datasets |
| Naive Bayes | Moderate | Yes (fastest) | No | Text classification, real-time |
| Gradient Boosting | Low-Moderate | Yes | No | Maximum accuracy, fraud, finance |
| Neural Network | Very Low | Yes | Mandatory | Images, text, complex patterns |


In [162]:
# ── COMPLETE MODEL COMPARISON — All Algorithms on Same Dataset ───────────────
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
import xgboost as xgb

np.random.seed(42)
from sklearn.datasets import make_classification
X_comp, y_comp = make_classification(n_samples=2000, n_features=12,
                                      n_informative=8, random_state=42)

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_comp, y_comp, test_size=0.2, random_state=42, stratify=y_comp)

scaler_c  = StandardScaler()
X_tr_c_sc = scaler_c.fit_transform(X_tr_c)
X_te_c_sc = scaler_c.transform(X_te_c)

algorithms = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN (K=7)':           KNeighborsClassifier(n_neighbors=7),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                                    random_state=42),
    'SVM (RBF)':           SVC(kernel='rbf', probability=True, random_state=42),
    'Naive Bayes':         GaussianNB(),
    'XGBoost':             xgb.XGBClassifier(n_estimators=200, eval_metric='logloss',
                                                                                          verbosity=0, random_state=42),
    'MLP Neural Net':      MLPClassifier(hidden_layer_sizes=(128,64),
                                         max_iter=200, random_state=42,
                                         early_stopping=True,
                                         validation_fraction=0.1,
                                         verbose=False)
}

comparison = {}
for name, model in algorithms.items():
    model.fit(X_tr_c_sc, y_tr_c)
    y_pred_c = model.predict(X_te_c_sc)
    y_prob_c = model.predict_proba(X_te_c_sc)[:, 1]
    comparison[name] = {
        'Accuracy':  round(accuracy_score(y_te_c, y_pred_c), 4),
        'Precision': round(precision_score(y_te_c, y_pred_c), 4),
        'Recall':    round(recall_score(y_te_c, y_pred_c), 4),
        'F1':        round(f1_score(y_te_c, y_pred_c), 4),
        'AUC-ROC':   round(roc_auc_score(y_te_c, y_prob_c), 4)
    }

comp_df = pd.DataFrame(comparison).T.sort_values('AUC-ROC', ascending=False)
print("="*70)
print("  SUPERVISED LEARNING — COMPLETE ALGORITHM COMPARISON")
print("  Prodigy Training Hub | Trainer: Kajola Gbenga | CEO")
print("="*70)
print(comp_df.to_string())
print(f"\nBest algorithm by AUC-ROC: {comp_df['AUC-ROC'].idxmax()}")
print(f"Best algorithm by F1:      {comp_df['F1'].idxmax()}")


  SUPERVISED LEARNING — COMPLETE ALGORITHM COMPARISON
  Prodigy Training Hub | Trainer: Kajola Gbenga | CEO
                     Accuracy  Precision  Recall      F1  AUC-ROC
MLP Neural Net         0.8975     0.8995   0.895  0.8972   0.9624
SVM (RBF)              0.8725     0.8706   0.875  0.8728   0.9542
KNN (K=7)              0.8825     0.8806   0.885  0.8828   0.9511
Random Forest          0.8850     0.8969   0.870  0.8832   0.9509
XGBoost                0.8700     0.8776   0.860  0.8687   0.9469
Naive Bayes            0.7600     0.7364   0.810  0.7714   0.8522
Decision Tree          0.7400     0.7051   0.825  0.7604   0.7825
Logistic Regression    0.6925     0.6954   0.685  0.6902   0.7685

Best algorithm by AUC-ROC: MLP Neural Net
Best algorithm by F1:      MLP Neural Net


In [163]:
# ── Visualise comparison ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
x      = np.arange(len(comp_df.index))
width  = 0.15

color_list = [COLORS['blue'], COLORS['teal'], COLORS['coral'],
              COLORS['amber'], COLORS['purple']]

for i, (metric, color) in enumerate(zip(metrics, color_list)):
    axes[0].bar(x + i*width, comp_df[metric], width=width,
                label=metric, color=color, alpha=0.85)

axes[0].set_xticks(x + width*2)
axes[0].set_xticklabels(comp_df.index, rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('Score')
axes[0].set_title('All Metrics by Algorithm')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].set_ylim(0.7, 1.02)

# AUC-ROC horizontal bar chart
comp_sorted = comp_df['AUC-ROC'].sort_values()
bar_colors  = [COLORS['blue'] if v < comp_sorted.max() else COLORS['teal']
               for v in comp_sorted.values]
axes[1].barh(comp_sorted.index, comp_sorted.values, color=bar_colors)
for i, v in enumerate(comp_sorted.values):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=10)
axes[1].set_xlabel('AUC-ROC')
axes[1].set_title('AUC-ROC Ranking')
axes[1].set_xlim(0.7, 1.03)

plt.suptitle('Algorithm Comparison Dashboard\nProdigy Training Hub | Trainer: Kajola Gbenga | CEO',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('algorithm_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()


---
## Summary & Next Steps

### What you have covered in this masterclass:
1. **Supervised Learning fundamentals** — regression vs classification, workflow, bias-variance tradeoff
2. **Linear Regression** — OLS, Ridge, Lasso, ElasticNet, coefficient interpretation
3. **Logistic Regression** — sigmoid, log-odds, AUC-ROC, confusion matrix
4. **K-Nearest Neighbors** — distance-based prediction, elbow method, scaling requirement
5. **Decision Trees** — Gini impurity, export rules, feature importance, overfitting control
6. **Random Forest** — bagging, OOB score, feature importance, hyperparameter tuning
7. **SVM** — maximum margin, kernel trick, C and gamma parameters
8. **Naive Bayes** — Bayes theorem, text classification, TF-IDF pipeline
9. **Gradient Boosting (XGBoost)** — sequential boosting, early stopping, imbalanced data handling
10. **Neural Networks** — MLP, backpropagation, activation functions, regularisation
11. **Model Evaluation** — all metrics, cross-validation, algorithm selection guide

### Recommended next topics:
- **Unsupervised Learning:** K-Means, DBSCAN, PCA, Autoencoders
- **Feature Engineering:** encoding, imputation, feature selection (RFE, SHAP)
- **Hyperparameter Optimization:** Optuna, Bayesian optimization
- **Handling Imbalanced Data:** SMOTE, class weights, threshold tuning
- **SHAP Values:** model explainability for any algorithm
- **Time Series:** ARIMA, Prophet, LSTM

---

> **Prodigy Training Hub**  
> Trainer: **Kajola Gbenga**  
> Title: CEO, Prodigy Training Hub  
> Programme: Data Science & Machine Learning Masterclass  

*This notebook is the intellectual property of Prodigy Training Hub. Designed for educational use within the Data Science & ML programme.*
